# Download CoNLL-2003

Paper (Tjong Kim Sang & De Meulder 2003). Data via Kaggle: https://www.kaggle.com/datasets/juliangarratt/conll2003-dataset 

Note: make sure your environment has `kagglehub` and `datasets`, and that Kaggle API credentials are configured (so the Kaggle download works on macOS/Windows too).

Original link from paper broken? 

## Kaggle Dataset

In [1]:
import kagglehub
from pathlib import Path
import shutil

# create root dir for data
root = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
out = root / "data" / "conll2003"
out.mkdir(parents=True, exist_ok=True)

# pull data from kaggle
path = kagglehub.dataset_download("juliangarratt/conll2003-dataset")
src = Path(path)
for n in ["eng.train", "eng.testa", "eng.testb"]:
    f = src / n if (src / n).exists() else next(src.rglob(n), None)
    if f:
        shutil.copy(f, out / n)

print("Original CoNLL-2003:", out)

Original CoNLL-2003: /home/arda/Documents/uva_msds/Sem4/DeepLearning/ds6050-project/data/conll2003


### Dataset verification: Load CoNLL-2003 via HuggingFace datasets, confirm train/dev/test sizes (14,987/3,466/3,684 sentence)

In [2]:
for name in ["eng.train", "eng.testa", "eng.testb"]:
    if (out / name).exists():
        count = sum(1 for line in (out / name).open() 
                    if line.strip() == "")
        print(f"{name}: {count} sentences")

# Each sentence is separated by a blank line, so the number of blank lines equals the number of sentences

eng.train: 14987 sentences
eng.testa: 3466 sentences
eng.testb: 3684 sentences


### Report entity-type distribution (PER, ORG, LOC, MISC counts),

In [3]:
import pandas as pd
from collections import Counter

def count_entities(filepath):
    counts = Counter()
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            # Skip empty lines and document separator
            if line == "" or line.startswith("-DOCSTART-"):
                continue
            
            # Extract the NER tag (the last column)
            ner_tag = line.split()[-1]
            
            # Ignore tokens that are not entities
            if ner_tag == "O":
                continue
            
            # Strip the B- or I- prefix ('B-PER' becomes 'PER')
            entity_type = ner_tag.split("-")[1]
            counts[entity_type] += 1
    return counts

# Container for row data
data_rows = []

for name, split in [("eng.train", "train"), 
                    ("eng.testa", "validation"), 
                    ("eng.testb", "test")]:
    
    # Run the counter for the current file
    counts = count_entities(out / name)
    
    # Add each entity count to list as a dictionary
    for entity, count in counts.items():
        data_rows.append({
            "Dataset": split,
            "Entity": entity,
            "Count": count
        })

# Convert list of dictionaries into a DataFrame
df = pd.DataFrame(data_rows)

# Pivot to see a side-by-side comparison
df = df.pivot(index="Entity", columns="Dataset", values="Count")

print(df)

Dataset  test  train  validation
Entity                          
LOC      1925   8297        2094
MISC      918   4593        1268
ORG      2496  10025        2092
PER      2773  11128        3149


### Ensure -DOCSTART- boundaries are preserved in Kaggle data

In [4]:
docstart_results = []

for name in ["eng.train", "eng.testa", "eng.testb"]:
    with open(out / name) as f:
        # Count occurrences of document separator
        count = sum(1 for line in f if line.startswith("-DOCSTART-"))
        docstart_results.append({"File": name, "DOCSTART_Count": count})

df_docstart = pd.DataFrame(docstart_results)
df_docstart

,File,DOCSTART_Count
0,eng.train,946
1,eng.testa,216
2,eng.testb,231


# Hugging Face Data

In [5]:
import os
os.environ["HF_DATASETS_CACHE"] = str(root / "data" / "hf_cache")

from datasets import load_dataset
dataset = load_dataset("conll2003", trust_remote_code=True)

In [6]:
for n in ["eng.train", "eng.testa", "eng.testb"]:
    if (out / n).exists():
        print(f"{n}: {sum(1 for _ in (out / n).open())} lines")
print("HF:", len(dataset["train"]), len(dataset["validation"]), len(dataset["test"]))

eng.train: 219554 lines
eng.testa: 55044 lines
eng.testb: 50350 lines
HF: 14041 3250 3453


### Incorrect train/dev/test split in HF data ^ 

### Confirm HF has no DOCSTART markers

In [7]:
# Confirm HF has no DOCSTART markers
docstart_count = sum(
    1 for ex in dataset["train"] if "-DOCSTART-" in ex["tokens"]
)
print(f"\n-DOCSTART- examples in HF train: {docstart_count}")


-DOCSTART- examples in HF train: 0
